# Newton fractal

*maths · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/newton_fractal)

Newton's method $z \leftarrow z - p(z)/p'(z)$ run on every pixel of the complex plane at once, as one complex tensor. Colouring each pixel by the root it reaches draws the basins of attraction, whose boundaries are fractal, while close to a root the error squares at every step.

**Vector form**

$$z_{n+1} = z_n - \frac{p(z_n)}{p'(z_n)},\qquad p(z) = \sum_{k=0}^{K-1} c_k z^k,\qquad z_n \to r_{j^*}$$

**Index (tensor) form**

$$p_p = c_k\,V_{pk},\quad V_{pk} = z_p^{\,k},\qquad c'_k = (k+1)\,c_{k+1},\qquad z_p \leftarrow z_p - \frac{c_k V_{pk}}{c'_k V_{pk}},\qquad j^*_p = \arg\min_j |z_p - r_j|$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $z_p$ | `grid(center, half_width, n, device)` | (H, W) complex | one starting point per pixel $p = (h, w)$: $z = x_w + i\,y_h$ |
| $c_k$ | `coefficients(name, device)` | (K,) | polynomial coefficients from the constant term up, e.g. $z^3 - 1 \to (-1, 0, 0, 1)$ |
| $V_{pk} = z_p^{\,k}$ | `powers(z, degree)` | (H, W, K) | powers $1, z, z^2, \dots$ of every pixel, built by a running product |
| $c_k V_{pk}$ | `einsum('...k,k->...', V, c)` | (H, W) | repeated $k$ ⇒ summed: $p(z)$ on every pixel |
| $c'_k = (k+1)\,c_{k+1}$ | `arange(1, K) * c[1:]` | (K-1,) | coefficients of $p'(z)$: differentiating shifts the powers down |
| $r_j$ | `linalg.eigvals(companion)` | (J,) | the roots, as eigenvalues of the companion matrix |
| $\vert z_p - r_j\vert $ | `(z[..., None] - roots).abs()` | (H, W, J) | distance of every pixel to every root; the minimum over $j$ gives the basin label |
| $e_{k+1}/e_k^2$ | `error_trace(z0, c)` | (n+1,) | one starting point in double precision: the ratio tends to $/p''(r)/2p'(r)/$ |

Expanding $p$ around a simple root $r$ gives $e_{k+1} \approx \frac{p''(r)}{2p'(r)}\,e_k^2$: the number of correct digits doubles each step, which the right panel shows as a parabola on a log scale (for $z^3 - 1$ the constant is exactly 1). Far from the roots Newton's method is chaotic. Near a critical point, where $p'(z) = 0$, a tiny step in $z$ sends the next iterate anywhere in the plane, so copies of the whole picture appear at every scale. The basins of a polynomial with three or more roots share one boundary, the Julia set of the Newton map: every boundary point of one basin is a boundary point of all of them (the Wada property). The terminal counts how many boundary pixels have every colour in their $3\times3$ neighbourhood, at full view and after zooming in. The share stays about the same at every scale. Try $z^3 - 2z + 2$, where starting points near 0 fall into the attracting cycle $0 \to 1 \to 0$ and never reach a root (grey regions). The whole grid is a single complex tensor, so one einsum updates every pixel.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/newton_fractal/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    name, n_iter, tol = params["polynomial"], params["iterations"], params["tol"]
    n = 96 if fast_mode() else params["pixels"]
    c = tensors.coefficients(name, dev)  # (K,)
    roots = tensors.roots(c)  # (J,)
    print(f"p(z) = {tensors.LABELS[name]}, roots " + ", ".join(f"{r.real:+.4f}{r.imag:+.4f}i" for r in roots.tolist()))

    half = params["extent"]
    z_full = tensors.grid(0j, half, n, dev)  # (H, W) complex: one starting point per pixel
    labels, conv = solve.iterate(z_full, c, roots, n_iter, tol)
    final = labels[-1].long()
    for j in range(len(roots)):
        print(f"basin of root {j}: {(final == j).float().mean().item():6.1%} of the pixels")
    print(f"not converged after {n_iter} steps: {(final < 0).float().mean().item():.1%} of the pixels")
    print(f"mean steps to converge: {conv[final >= 0].float().mean().item():.1f}")

    center = solve.zoom_center(z_full, final, len(roots), half)
    z_zoom = tensors.grid(center, half / params["zoom"], n, dev)
    labels_zoom, conv_zoom = solve.iterate(z_zoom, c, roots, n_iter, tol)
    print(f"zoom ×{params['zoom']:.0f} around {center.real:+.4f}{center.imag:+.4f}i")
    for tag, lab in (("full view", final), ("zoom", labels_zoom[-1].long())):
        b, w = solve.boundary_stats(lab, len(roots))
        print(f"{tag:9s}: {b:5.1%} of pixels on a boundary, {w:5.1%} of those touch all {len(roots)} basins")

    errors, root = solve.error_trace(complex(roots[0].item()) + 0.45 + 0.3j, c)
    limit = solve.newton_constant(root, c)
    print(f"one start point → root {root.real.item():+.4f}{root.imag.item():+.4f}i, |p''/(2p')| = {limit:.4f}")
    for k in range(len(errors) - 1):
        if errors[k + 1] < 1e-14:
            break
        print(f"  step {k + 1}: error {errors[k + 1]:.3e},  e_(k+1) / e_k² = {errors[k + 1] / errors[k] ** 2:.4f}")

    views = [(labels, conv, half, 0j), (labels_zoom, conv_zoom, half / params["zoom"], center)]
    return plot.animate(views, len(roots), errors, limit, tensors.LABELS[name], out_dir / "newton_fractal.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/newton_fractal/equation.py
r"""Newton's method → tensor translation.

Every pixel p of the complex plane holds one starting point z_p, so the whole picture is one complex tensor.
A polynomial with coefficients c_k (lowest power first) is a contraction with the powers of z,

    p(z_p) = \sum_k c_k V_{pk},        V_{pk} = z_p^k        (k summed),

and its derivative has the shifted coefficients c'_k = (k + 1) c_{k+1}. Newton's step then updates every pixel at once,

    z_p \leftarrow z_p - p(z_p) / p'(z_p),

and the root a pixel has reached is the nearest one, j^*_p = \arg\min_j |z_p - r_j|.
"""

import torch


def powers(z: torch.Tensor, degree: int) -> torch.Tensor:
    r"""V_{pk} = z_p^k for k = 0..degree, built by a running product (exact, unlike z ** k).   (...) → (..., K)"""
    ones = torch.ones_like(z)[..., None]
    return torch.cumprod(torch.cat([ones, z[..., None].expand(*z.shape, degree)], dim=-1), dim=-1)


def poly(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    r"""p(z_p) = c_k V_{pk}   (k summed).   z: (...), c: (K,) → (...)"""
    return torch.einsum("...k,k->...", powers(z, len(c) - 1), c)


def derivative(c: torch.Tensor) -> torch.Tensor:
    r"""c'_k = (k + 1) c_{k+1}: coefficients of p'(z).   (K,) → (K-1,)"""
    k = torch.arange(1, len(c), device=c.device)
    return k * c[1:]


def newton_step(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    r"""z ← z − p(z)/p'(z) on every pixel at once.   (...) → (...)"""
    return z - poly(z, c) / poly(z, derivative(c))


def nearest_root(z: torch.Tensor, roots: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Distance to the nearest root min_j |z_p − r_j| and its index j^*_p.   z: (...), roots: (J,) → (...), (...)"""
    dist = (z[..., None] - roots).abs()  # (..., J)
    return dist.min(dim=-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/newton_fractal/tensors.py
"""Build the concrete tensors: the polynomial's coefficients, its roots and the complex pixel grids."""

import math

import torch

# Coefficients c_k from the constant term up (lowest power first), all monic.
POLYNOMIALS = {
    "z3_minus_1": [-1.0, 0.0, 0.0, 1.0],  # z³ − 1: the three cube roots of unity
    "z4_minus_1": [-1.0, 0.0, 0.0, 0.0, 1.0],  # z⁴ − 1: ±1, ±i
    "z3_minus_2z_plus_2": [2.0, -2.0, 0.0, 1.0],  # z³ − 2z + 2: has an attracting 2-cycle 0 → 1 → 0
}
LABELS = {"z3_minus_1": "z³ − 1", "z4_minus_1": "z⁴ − 1", "z3_minus_2z_plus_2": "z³ − 2z + 2"}


def coefficients(name: str, device: torch.device, dtype=torch.complex64) -> torch.Tensor:
    """c_k, shape (K,) = (degree + 1,)."""
    return torch.tensor(POLYNOMIALS[name], dtype=dtype, device=device)


def roots(c: torch.Tensor) -> torch.Tensor:
    """The roots r_j as eigenvalues of the companion matrix (works for any monic polynomial): (J,)."""
    n = len(c) - 1
    companion = torch.zeros(n, n, dtype=torch.complex128)
    companion[1:, :-1] = torch.eye(n - 1)  # ones on the sub-diagonal
    companion[:, -1] = -c[:-1].to(torch.complex128).cpu()
    r = torch.linalg.eigvals(companion)
    order = torch.argsort(torch.atan2(r.imag, r.real) % (2 * math.pi))  # sorted by angle: stable colours
    return r[order].to(c.dtype).to(c.device)


def grid(center: complex, half_width: float, n: int, device: torch.device) -> torch.Tensor:
    """Complex pixel grid z_{hw} = x_w + i y_h of n × n pixels around center: (H, W). n even, so 0 is never a pixel."""
    t = torch.linspace(-half_width, half_width, n, device=device)
    x = center.real + t  # (W,)
    y = center.imag - t  # (H,): the top row has the largest imaginary part
    return torch.complex(x[None, :].expand(n, n), y[:, None].expand(n, n))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/newton_fractal/solve.py
"""Run Newton's method on whole grids, record which root every pixel reaches and when, and measure the basins."""

import torch
import torch.nn.functional as F

from problems._common.progress import steps

from .equation import derivative, nearest_root, newton_step, poly


def iterate(z0: torch.Tensor, c: torch.Tensor, roots: torch.Tensor, n_iter: int, tol: float):
    """Newton on every pixel for n_iter steps.

    Returns labels (n_iter+1, ...) int8, the root each pixel is within tol of at step k (or −1), and
    conv_iter (...), the first step at which it got there (n_iter + 1 if it never did).
    """
    labels = torch.empty((n_iter + 1, *z0.shape), dtype=torch.int8, device=z0.device)
    z = z0
    for k in steps(n_iter + 1, "Newton steps"):
        dist, j = nearest_root(z, roots)
        labels[k] = torch.where(dist < tol, j, -1)  # NaN (from p' = 0) compares False: not converged
        if k < n_iter:
            z = newton_step(z, c)
    converged = labels >= 0
    first = converged.to(torch.uint8).argmax(dim=0)  # index of the first True
    conv_iter = torch.where(converged.any(dim=0), first, n_iter + 1)
    return labels, conv_iter


def error_trace(z0: complex, c: torch.Tensor, n_iter: int = 12) -> tuple[torch.Tensor, torch.Tensor]:
    """Follow one starting point in double precision: errors e_k = |z_k − r| to the root r it converges to."""
    c = c.to(torch.complex128).cpu()
    z = torch.tensor([z0], dtype=torch.complex128)
    path = [z]
    for _ in range(n_iter):
        z = newton_step(z, c)
        path.append(z)
    path = torch.cat(path)  # (n_iter+1,)
    root = path[-1]
    return (path - root).abs(), root


def newton_constant(root: torch.Tensor, c: torch.Tensor) -> float:
    r"""|p''(r) / (2 p'(r))|: the limit of e_{k+1}/e_k² (quadratic convergence)."""
    c = c.to(torch.complex128).cpu()
    d1 = derivative(c)
    return (poly(root, derivative(d1)) / (2 * poly(root, d1))).abs().item()


def roots_present(label: torch.Tensor, n_roots: int) -> torch.Tensor:
    """Which roots occur in the 3×3 neighbourhood of every pixel: (H, W) labels → (J, H, W) booleans."""
    one_hot = torch.stack([(label == j).float() for j in range(n_roots)])  # (J, H, W)
    return F.max_pool2d(one_hot[None], kernel_size=3, stride=1, padding=1)[0] > 0


def boundary_stats(label: torch.Tensor, n_roots: int) -> tuple[float, float]:
    """(fraction of pixels on a basin boundary, fraction of those boundary pixels that touch every basin)."""
    count = roots_present(label, n_roots).sum(dim=0)  # (H, W): number of different basins nearby
    boundary = count >= 2
    wada = count == n_roots
    return boundary.float().mean().item(), (wada.sum() / boundary.sum().clamp(min=1)).item()


def zoom_center(z: torch.Tensor, label: torch.Tensor, n_roots: int, half_width: float) -> complex:
    """A boundary pixel where all basins meet (a Wada point): the one nearest a fixed off-axis spot."""
    wada = roots_present(label, n_roots).all(dim=0)
    target = complex(0.3 * half_width, 0.45 * half_width)
    dist = torch.where(wada, (z - target).abs(), torch.inf)
    return complex(z.flatten()[dist.argmin()].item())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/newton_fractal/plot.py
"""Animate the basins of attraction filling in, a zoom onto their boundary, and the error of one starting point."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import to_rgb

from problems._common.plotting import ACCENT, frame_indices, save_gif

GREY = np.array(to_rgb("#e5e7eb"))


def colour(labels: np.ndarray, conv: np.ndarray, k: int, n_roots: int) -> np.ndarray:
    """RGB image at step k: the root's colour, darker the more steps it took; grey while not yet converged."""
    base = np.array([to_rgb(ACCENT[j % len(ACCENT)]) for j in range(n_roots)])  # (J, 3)
    final = labels[-1].astype(int)
    shade = 1.0 - 0.75 * np.minimum(conv, 12)[..., None] / 12  # (H, W, 1)
    done = ((conv <= k) & (final >= 0))[..., None]
    return np.where(done, base[np.maximum(final, 0)] * shade, GREY)


def animate(views: list, n_roots: int, errors: torch.Tensor, limit: float, label: str, path: Path) -> Path:
    """views: [(labels (K+1, H, W), conv (H, W), half_width, center), ...] for the full plane and the zoom."""
    data = [(lab.cpu().numpy(), conv.cpu().numpy(), half, ctr) for lab, conv, half, ctr in views]
    n_steps = data[0][0].shape[0]
    e = errors.numpy()
    e = e[: int(np.argmax(e < 1e-15)) or len(e)]  # stop once the error reaches round-off

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8), gridspec_kw={"width_ratios": [1, 1, 1.15]})
    images = []
    for ax, (lab, conv, half, ctr), name in zip(axes, data, ["complex plane", "zoom onto a boundary point"]):
        ext = [ctr.real - half, ctr.real + half, ctr.imag - half, ctr.imag + half]
        images.append(ax.imshow(colour(lab, conv, 0, n_roots), extent=ext, interpolation="nearest"))
        ax.set_title(name, fontsize=10)
        ax.set_xlabel("Re z")
        ax.grid(False)
    axes[0].set_ylabel("Im z")
    zc, zh = data[1][3], data[1][2]
    axes[0].add_patch(plt.Rectangle((zc.real - zh, zc.imag - zh), 2 * zh, 2 * zh, fill=False, ec="#111827", lw=1.2))

    ax = axes[2]
    ks = np.arange(len(e))
    ax.semilogy(ks, e, "o-", color="#2563eb", ms=4, lw=1.2, label="error $e_k$")
    start = min(2, len(e) - 1)  # the squaring law holds once the point is close: start it at step 2
    guide = [e[start]]
    for _ in ks[start + 1 :]:
        guide.append(limit * guide[-1] ** 2)
    ax.semilogy(
        ks[start:], np.maximum(guide, 1e-17), "--", color="#9ca3af", lw=1, label=r"$e_{k+1} = |p''/2p'|\,e_k^2$"
    )
    ax.set_ylim(1e-16, 2)
    ax.set_xlabel("Newton step $k$")
    ax.set_title("quadratic convergence: digits double", fontsize=10)
    ax.legend(fontsize=8, loc="lower left")
    (dot,) = ax.plot([], [], "o", color="#dc2626", ms=8)
    title = fig.suptitle("", fontsize=11)

    def update(k: int):
        for img, (lab, conv, _, _) in zip(images, data):
            img.set_data(colour(lab, conv, k, n_roots))
        j = min(k, len(e) - 1)
        dot.set_data([j], [e[j]])
        title.set_text(f"Newton's method for p(z) = {label}, step {k}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_steps, 120), path, fps=5)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'polynomial': 'z3_minus_1',
    'pixels': 320,
    'extent': 1.5,
    'iterations': 40,
    'zoom': 20.0,
    'tol': 0.0001
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.newton_fractal")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.newton_fractal.main")
problem_dir = Path("problems/maths/newton_fractal")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
